# 99 — Subir proyecto v2 a Google Drive

Ejecutá este notebook **una vez** después de cambiar código en `d10sformer-v2` local o en Git.

Destino: `MyDrive/d10sformer-v2` (id `1Xz1rbw8t8jF_6J5Ez-_vUb7MuPG69w-O`)

No sube `data/` (los datos quedan en `MyDrive/d10sformer`).

In [ ]:
!pip install -q google-api-python-client google-auth-oauthlib

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from pathlib import Path
import mimetypes

V2_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
V2_FOLDER_ID = '1Xz1rbw8t8jF_6J5Ez-_vUb7MuPG69w-O'

# Opción A: clonar desde GitHub (recomendado tras push)
REPO = 'https://github.com/jose-marin-db/udesa-nlp-futbol-D10Sformer.git'
!test -d /content/d10sformer-b || git clone -q {REPO} /content/d10sformer-b
SOURCE = Path('/content/d10sformer-b')
if not (SOURCE / 'src' / 'paths.py').exists():
    SOURCE = V2_ROOT  # fallback: solo re-subir lo que ya está en Drive

service = build('drive', 'v3')
print('SOURCE =', SOURCE)
print('V2_ROOT =', V2_ROOT)

In [ ]:
def ensure_folder(parent_id: str, name: str, cache: dict) -> str:
    key = (parent_id, name)
    if key in cache:
        return cache[key]
    q = f"'{parent_id}' in parents and name = '{name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    res = service.files().list(q=q, fields='files(id)').execute()
    if res.get('files'):
        fid = res['files'][0]['id']
    else:
        meta = {'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]}
        fid = service.files().create(body=meta, fields='id').execute()['id']
    cache[key] = fid
    return fid


def upload_file(local: Path, parent_id: str):
    q = f"'{parent_id}' in parents and name = '{local.name}' and trashed = false"
    existing = service.files().list(q=q, fields='files(id)').execute().get('files', [])
    mime, _ = mimetypes.guess_type(str(local))
    media = MediaFileUpload(str(local), mimetype=mime or 'application/octet-stream', resumable=True)
    if existing:
        service.files().update(fileId=existing[0]['id'], media_body=media).execute()
        return 'updated'
    body = {'name': local.name, 'parents': [parent_id]}
    service.files().create(body=body, media_body=media).execute()
    return 'created'


SKIP_DIRS = {'data', '.git', '__pycache__', '.ipynb_checkpoints', 'checkpoints', 'reports'}
UPLOAD_EXT = {'.py', '.ipynb', '.md', '.yaml', '.txt', '.json'}

cache = {}
count = 0
for path in sorted(SOURCE.rglob('*')):
    if not path.is_file():
        continue
    if path.suffix not in UPLOAD_EXT:
        continue
    rel = path.relative_to(SOURCE)
    if rel.parts and rel.parts[0] in SKIP_DIRS:
        continue
    if 'data' in rel.parts:
        continue
    parent = V2_FOLDER_ID
    for part in rel.parts[:-1]:
        parent = ensure_folder(parent, part, cache)
    status = upload_file(path, parent)
    count += 1
    print(f'{status:7} {rel}')

print(f'\n✓ {count} archivos sincronizados a d10sformer-v2')